# 03 · Ranking, ROC/AUC y umbral — inicio para completar

Distinguir calidad del ranking de una decisión por umbral y elegir ese umbral sin mirar test.

**Ideas que aparecen:** laboratorio 02; proporción y probabilidad (cápsula 5). Puedes consultar las cápsulas
opcionales de `ruta/Puentes de entrada.md` cuando alguna te haga falta.

Datos **sintéticos educativos**, creados en este repositorio y dedicados a
CC0-1.0. No representan personas ni un rendimiento oficial IOAI.
Todo el ejercicio usa CPU y archivos locales; no requiere cuentas ni red.

El inicio ya corre. Las celdas de experimentación son lugares para cambiar una idea y observar qué ocurre; la solución está en otro archivo si quieres contrastarla.
Puedes recorrerlo en una o varias sesiones. No hay límite de juez ni
obligación de completar todos los experimentos para abrir el siguiente cuaderno.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATOS = Path("datos")
assert DATOS.is_dir(), "Abre el notebook desde su carpeta: deben existir datos/ e inicio.ipynb."
SEMILLA = 17


## Dos preguntas diferentes
Aquí el modelo ya produjo un `score` entre 0 y 1. Un score mayor
significa más evidencia de clase 1; no suponemos que esté calibrado
como probabilidad. `score >= umbral` convierte ranking en decisión.
Precision = TP/(TP+FP), recall = TP/(TP+FN), F1 es su media armónica.
Cuando no hay positivos predichos usamos `zero_division=0`.

La curva ROC recorre tasa de falsos positivos (FP/(FP+TN)) y recall
al variar el umbral. ROC-AUC evalúa orden: probabilidad de que un
positivo elegido al azar tenga score mayor que un negativo, con
medio crédito para empates. No cambia al mover el umbral de decisión.
Con una sola clase, esa comparación no existe: reporta «no definida».
Con fuerte desbalance conviene además mirar precision y recall:
una AUC alta no garantiza que cada alerta sea útil.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
val = pd.read_csv(DATOS / "validacion.csv")
print("Balance:", val.etiqueta.value_counts().to_dict())
print("Baseline siempre 0, accuracy:", accuracy_score(val.etiqueta, np.zeros(len(val))))
umbrales = np.arange(5, 96, 5) / 100  # grilla definida antes de evaluar


## Zona de experimentación
Elige un umbral usando F1 de validación. Conserva 0.5 como baseline.
En empate elige el umbral más alto para que el procedimiento sea
reproducible. Implementa AUC mediante comparaciones positivo-negativo
y contrástala con scikit-learn. No uses etiquetas de transferencia
para escoger el umbral.


In [ ]:
def elegir_umbral(y, scores, candidatos):
    # TRABAJO: evaluar la grilla usando solo validación.
    return 0.5

def auc_por_pares(y, scores):
    # TRABAJO: sustituye esta llamada por comparaciones, incluido empate.
    if len(np.unique(y)) < 2:
        return None
    return float(roc_auc_score(y, scores))


## Comparación en las mismas condiciones
El baseline F1 usa umbral 0.5 sobre la misma validación. Exploramos cuánto cambia F1 sin cambiar el ranking. La figura y la tabla de puntos ROC muestran la misma idea desde dos representaciones. Transferencia usa el umbral ya elegido.


In [ ]:
elegido = elegir_umbral(val.etiqueta.to_numpy(), val.score.to_numpy(), umbrales)
pred = val.score.to_numpy() >= elegido
fpr, tpr, cortes = roc_curve(val.etiqueta, val.score)
print(pd.DataFrame({"falsos_positivos": fpr, "recall": tpr, "umbral": cortes}))
print("Matriz [[TN,FP],[FN,TP]]:", confusion_matrix(val.etiqueta, pred, labels=[0, 1]))
resultado = {
    "baseline": float(f1_score(val.etiqueta, val.score >= 0.5, zero_division=0)),
    "validacion": float(f1_score(val.etiqueta, pred, zero_division=0)),
    "umbral": float(elegido), "auc": auc_por_pares(val.etiqueta, val.score),
    "precision": float(precision_score(val.etiqueta, pred, zero_division=0)),
    "recall": float(recall_score(val.etiqueta, pred, zero_division=0)),
}
assert abs(resultado["auc"] - roc_auc_score(val.etiqueta, val.score)) < 1e-12
assert auc_por_pares([0, 1], [0.4, 0.4]) == 0.5
assert auc_por_pares([0, 0], [0.1, 0.9]) is None


<details><summary>Idea · decisión</summary>Para cada candidato, transforma scores en etiquetas con >= y calcula F1.</details>
<details><summary>Idea · ranking</summary>Forma una matriz de diferencias: scores_positivos[:, None] - scores_negativos[None, :].</details>
<details><summary>Idea · empate</summary>Promedia 1 si la diferencia es positiva, 0.5 si es cero y 0 si es negativa.</details>


## Transferencia: decide antes de mirar el resultado
Congela el umbral elegido y mide un conjunto con otra prevalencia. Describe qué cambió en precision/recall y por qué no debes retocar el umbral mirando estas respuestas. Prueba una transformación estrictamente creciente de los scores: el orden y la AUC se conservan, el umbral numérico debe transformarse si quieres las mismas decisiones.


In [ ]:
nuevo = pd.read_csv(DATOS / "transferencia.csv")
pred_nuevo = nuevo.score >= elegido
resultado["transferencia"] = float(f1_score(nuevo.etiqueta, pred_nuevo, zero_division=0))
resultado["baseline_transferencia"] = float(f1_score(nuevo.etiqueta, nuevo.score >= 0.5, zero_division=0))
resultado["precision_transferencia"] = float(precision_score(nuevo.etiqueta, pred_nuevo, zero_division=0))
resultado["recall_transferencia"] = float(recall_score(nuevo.etiqueta, pred_nuevo, zero_division=0))
assert np.isclose(roc_auc_score(nuevo.etiqueta, nuevo.score), roc_auc_score(nuevo.etiqueta, nuevo.score ** 3))
assert np.array_equal(nuevo.score >= elegido, nuevo.score ** 3 >= elegido ** 3)


## Mirar la idea
Compara el dibujo con lo que esperabas antes de ejecutar.


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(fpr, tpr, marker=".", color="#287da3", label=f"ROC; AUC={resultado['auc']:.3f}")
ejes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", label="referencia sin orden útil")
tn, fp, fn, tp = confusion_matrix(val.etiqueta, pred, labels=[0, 1]).ravel()
ejes[0].scatter([fp / (fp + tn)], [tp / (tp + fn)], color="#d2604d", s=70, zorder=3, label="umbral elegido")
ejes[0].set(xlabel="tasa de falsos positivos", ylabel="recall", title="Ranking: todos los umbrales")
ejes[0].legend(fontsize=9)
f1s = [f1_score(val.etiqueta, val.score >= t, zero_division=0) for t in umbrales]
ejes[1].plot(umbrales, f1s, marker="o", color="#287da3")
ejes[1].axvline(0.5, color="gray", linestyle="--", label="baseline 0.5")
ejes[1].axvline(elegido, color="#d2604d", label=f"elegido {elegido:.2f}")
ejes[1].set(xlabel="umbral", ylabel="F1", title="Decisión: un umbral concreto", ylim=(0, 1.05))
ejes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()


## Para seguir explorando
¿Puedes mejorar F1 sin cambiar una sola posición del ranking? Mira cómo se mueve el punto de operación al cambiar el umbral. ¿Por qué un ranking muy bueno puede generar muchas falsas alarmas cuando hay pocos positivos?


## Comprobaciones del ejemplo
Estas aserciones detectan errores técnicos en el cuaderno y en su solución de referencia. No son un examen ni una escala de capacidad.


In [ ]:
assert isinstance(resultado, dict)


## Resultado reproducible
Esta celda guarda automáticamente las medidas para comprobar el material. Puedes conservar una copia de tu notebook y tus propias notas; no hay un formulario que rellenar.


In [ ]:
resultado.update({"laboratorio": '03_metricas', "version": 'inicio para completar',
                  "datos": "sintéticos CC0-1.0",
                  "metrica": 'F1 binario de clase positiva 1; ROC-AUC como diagnóstico de ranking (mayor es mejor)', "split": '60 pares etiqueta-score de validación; 50 de transferencia separados; modelo ya congelado'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
